# 06 · A wider model comparison

Notebooks 02 to 04 compare three baselines with LightGBM. This notebook adds four classification models from the course, on the same task (will a routine inspection fail?), with the same 17 inputs, the same time splits and the same metrics:

- ridge logistic regression (L2 penalty) and LASSO logistic regression (L1 penalty), Lectures 3 and 4
- random forest, Lecture 6 and HW4
- XGBoost, Lectures 6 and 15

Lecture 4 teaches ridge and LASSO as regression models. On a 0/1 label their output is not a probability, so we use the classification versions: logistic regression with the same penalties. The existing logistic regression (B2) is already the ridge version at `C = 1`. The ridge row therefore measures what tuning the penalty adds.

The protocol is the one the repo already uses. Settings are chosen on the validation year (2023), written to a frozen file, and only then are the models refitted on 2016 to 2023 and scored once on the test years. These models were added after the first test scoring, so their test rows carry the same caveat as M3. The final model is not re-picked from this table.

- Inputs: `data/processed/model_table.parquet` and the saved predictions of B0, B1, B2 and M1 in `preds/`.
- Outputs: `results/comparison_*.csv`, `results/comparison_selected_settings.json`, and `preds/C*_val.csv` and `preds/C*_test.csv`.
- Nothing from notebooks 01 to 05 is overwritten.

In [1]:
import sys
sys.path.insert(0, "..")

import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from src import compare as C
from src import evaluate as E
from src import models as M

pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 80)

BASELINES = ["B0_random", "B1_past_fail_rate", "B2_logreg", "M1_lgbm"]   # already scored in notebooks 02 to 04
NEW = list(C.GRIDS)
FROZEN_M1 = json.load(open("../docs/FROZEN_CONFIG.json"))
assert FROZEN_M1["history_features"] == M.HISTORY_FEATURES, "feature list changed after the freeze"


def saved(name, rows, split):
    """Saved predictions of an existing model, checked to be in the same row order as `rows`."""
    preds = pd.read_csv(f"../preds/{name}_{split}.csv", dtype={"inspection_id": str})
    assert (preds["inspection_id"].to_numpy() == rows["inspection_id"].to_numpy()).all(), f"{name}: rows differ"
    return preds["score"].to_numpy()

In [2]:
table = M.load_model_table("../data/processed/model_table.parquet")
train, val = table[table["split"] == "train"], table[table["split"] == "val"]
y_val, val_groups = val[M.TARGET].to_numpy(), val["licenseno"].to_numpy()
print(f"train {len(train):,} rows (2016 to 2022) | validation {len(val):,} rows (2023) | "
      f"{M.linear_matrix(train).shape[1]} columns after one-hot encoding")

# Two checks that this notebook sees the same data, prepared the same way, as notebooks 02 and 03.
ridge_at_1 = C.score(C.fit("C1_ridge_logreg", train, {"C": 1}), val)
assert np.allclose(ridge_at_1, saved("B2_logreg", val, "val")), "ridge at C=1 should be B2"
lgbm_keys = ["num_leaves", "learning_rate", "n_estimators"]
m1 = M.fit_lgbm(train, M.HISTORY_FEATURES, {k: FROZEN_M1["M1_lgbm"][k] for k in lgbm_keys})
assert np.allclose(m1.predict_proba(val[M.HISTORY_FEATURES])[:, 1], saved("M1_lgbm", val, "val")), "M1 does not reproduce"
print("checks passed: ridge at C=1 reproduces B2, and M1 refitted from the frozen settings reproduces its saved predictions")

train 35,872 rows (2016 to 2022) | validation 5,521 rows (2023) | 68 columns after one-hot encoding


checks passed: ridge at C=1 reproduces B2, and M1 refitted from the frozen settings reproduces its saved predictions


## 1 · Choosing settings on the validation year

Each model gets a small grid, modeled on the course's grids. For every setting the model is fitted on 2016 to 2022 and scored on 2023. We keep the setting with the best hit rate in the first 20% of visits. Ties go to the higher AUC, then to the simpler setting. LightGBM was tuned the same way, over 12 settings (notebook 03).

`C` is the inverse of the penalty strength: a small `C` is a strong penalty. For XGBoost, `n_estimators` is an upper limit, and early stopping on the validation AUC decides how many trees are used, as for LightGBM.

In [3]:
grids, selected = [], {}
for name in NEW:
    grid, best = C.grid_search(name, train, val)
    grids.append(grid)
    selected[name] = {"params": best, "fixed": C.FIXED[name], "settings_tried": len(grid)}
    print(f"{C.LABELS[name]}: {len(grid)} settings tried, validation hit rate from "
          f"{grid['precision_at_k'].min():.1%} to {grid['precision_at_k'].max():.1%} | selected {best}")
grid_table = pd.concat(grids, ignore_index=True)
grid_table.to_csv("../results/comparison_grid_val.csv", index=False)

settings = {
    "status": "added after the first test evaluation; settings chosen on validation (2023) only; the final model is not re-picked from this comparison",
    "seed": M.SEED,
    "selection_metric": "precision_at_k on validation (2023); ties go to the higher AUC, then to the simpler setting",
    "features": "the 17 history features of docs/FROZEN_CONFIG.json, as the one-hot matrix of models.linear_matrix",
    "fit_rows_for_test": "train + validation (2016 to 2023), as for every other model",
    **selected,
}
json.dump(settings, open("../results/comparison_selected_settings.json", "w"), indent=2)
grid_table.round(4)

Ridge logistic regression: 6 settings tried, validation hit rate from 88.7% to 88.9% | selected {'C': 0.001}


LASSO logistic regression: 6 settings tried, validation hit rate from 88.9% to 89.3% | selected {'C': 0.001}


Random forest: 12 settings tried, validation hit rate from 89.6% to 90.5% | selected {'n_estimators': 500, 'max_depth': None, 'max_features': 'sqrt'}


XGBoost: 12 settings tried, validation hit rate from 88.6% to 90.9% | selected {'max_depth': 4, 'learning_rate': 0.05, 'n_estimators': 139}


,model,C,auc,catch_at_k,precision_at_k,n,n_estimators,max_depth,max_features,learning_rate,best_iteration
0,C1_ridge_logreg,0.001,0.7784,0.3189,0.8895,5521,NaN,NaN,NaN,NaN,NaN
1,C1_ridge_logreg,0.010,0.7775,0.3180,0.8868,5521,NaN,NaN,NaN,NaN,NaN
2,C1_ridge_logreg,0.100,0.7763,0.3189,0.8895,5521,NaN,NaN,NaN,NaN,NaN
3,C1_ridge_logreg,1.000,0.7761,0.3189,0.8895,5521,NaN,NaN,NaN,NaN,NaN
4,C1_ridge_logreg,10.000,0.7761,0.3189,0.8895,5521,NaN,NaN,NaN,NaN,NaN
5,C1_ridge_logreg,100.000,0.7761,0.3189,0.8895,5521,NaN,NaN,NaN,NaN,NaN
6,C2_lasso_logreg,0.001,0.7638,0.3202,0.8931,5521,NaN,NaN,NaN,NaN,NaN
7,C2_lasso_logreg,0.010,0.7784,0.3189,0.8895,5521,NaN,NaN,NaN,NaN,NaN
8,C2_lasso_logreg,0.100,0.7768,0.3189,0.8895,5521,NaN,NaN,NaN,NaN,NaN
9,C2_lasso_logreg,1.000,0.7763,0.3189,0.8895,5521,NaN,NaN,NaN,NaN,NaN


In [4]:
# Refit each model at its selected setting, save its validation predictions, and compare with the baselines.
val_scores = {name: saved(name, val, "val") for name in BASELINES}
for name in NEW:
    model = C.fit(name, train, selected[name]["params"])
    val_scores[name] = C.score(model, val)
    M.save_preds(val, val_scores[name], f"../preds/{name}_val.csv")
    best_in_grid = grid_table[grid_table["model"] == name]["precision_at_k"].max()
    assert np.isclose(E.evaluate(y_val, val_scores[name])["precision_at_k"], best_in_grid), f"{name}: refit differs from the grid"
    if name == "C2_lasso_logreg":
        weights = model[-1].coef_.ravel()
        print(f"LASSO keeps {int((weights != 0).sum())} of {weights.size} weights at the selected C (fit on 2016 to 2022)")
        assert (weights == 0).any(), "the L1 penalty did not apply: scikit-learn 1.8 or later is required"

val_results = E.results_table(y_val, val_scores, groups=val_groups)
val_results.to_csv("../results/comparison_val.csv", index=False)
val_diffs = pd.concat([E.difference_table(y_val, val_scores, name, ["M1_lgbm", "B2_logreg"], groups=val_groups) for name in NEW],
                      ignore_index=True)
val_diffs.insert(0, "split", "val")
print(val_results[["model", "precision_at_k", "precision_at_k_lo", "precision_at_k_hi", "catch_at_k", "auc"]].round(3).to_string(index=False))
print()
print(val_diffs[val_diffs["metric"] == "precision_at_k"].drop(columns=["split", "metric"]).round(4).to_string(index=False))

LASSO keeps 17 of 77 weights at the selected C (fit on 2016 to 2022)


            model  precision_at_k  precision_at_k_lo  precision_at_k_hi  catch_at_k   auc
        B0_random           0.558              0.543              0.572       0.200 0.500
B1_past_fail_rate           0.804              0.777              0.829       0.288 0.717
        B2_logreg           0.889              0.867              0.910       0.319 0.776
          M1_lgbm           0.905              0.884              0.924       0.324 0.780
  C1_ridge_logreg           0.889              0.870              0.910       0.319 0.778
  C2_lasso_logreg           0.893              0.873              0.915       0.320 0.764
 C3_random_forest           0.905              0.885              0.924       0.324 0.780
       C4_xgboost           0.909              0.886              0.928       0.326 0.785

           model     minus  difference   ci_lo   ci_hi      verdict
 C1_ridge_logreg   M1_lgbm     -0.0154 -0.0280 -0.0009   borderline
 C1_ridge_logreg B2_logreg      0.0000 -0.0091  0.010

## 2 · Freeze, then test

The test half below runs only once the selected settings have been copied to `docs/FROZEN_CONFIG_COMPARISON.json` and committed. It then checks that the frozen file still equals what the grid search selects, refits every new model on 2016 to 2023 and scores the test years (2024 to November 2026) once.

The rows for B0, B1, B2 and M1 come from their saved test predictions and must equal `results/results.csv`.

In [5]:
FROZEN_PATH = Path("../docs/FROZEN_CONFIG_COMPARISON.json")
FROZEN = json.load(open(FROZEN_PATH)) if FROZEN_PATH.exists() else None
if FROZEN is None:
    print("Not frozen yet. Copy results/comparison_selected_settings.json to docs/FROZEN_CONFIG_COMPARISON.json, "
          "commit it, and rerun this notebook to score the test years.")
else:
    assert FROZEN == json.load(open("../results/comparison_selected_settings.json")), "settings changed after the freeze"
    fit_rows, test = table[table["split"].isin(["train", "val"])], table[table["split"] == "test"]
    assert fit_rows["date"].max() < pd.Timestamp("2024-01-01") <= test["date"].min()      # nothing is fitted on the test period
    y_test, test_groups = test[M.TARGET].to_numpy(), test["licenseno"].to_numpy()

    test_scores = {name: saved(name, test, "test") for name in BASELINES}
    for name in NEW:
        model = C.fit(name, fit_rows, FROZEN[name]["params"])
        test_scores[name] = C.score(model, test)
        M.save_preds(test, test_scores[name], f"../preds/{name}_test.csv")
        if name == "C2_lasso_logreg":
            weights = model[-1].coef_.ravel()
            print(f"LASSO keeps {int((weights != 0).sum())} of {weights.size} weights when refitted on 2016 to 2023")

    test_results = E.results_table(y_test, test_scores, groups=test_groups)
    published = pd.read_csv("../results/results.csv").set_index("model")
    for name in BASELINES:
        ours = test_results.set_index("model").loc[name]
        assert np.allclose(ours[["auc", "catch_at_k", "precision_at_k"]].astype(float),
                           published.loc[name, ["auc", "catch_at_k", "precision_at_k"]].astype(float)), name
    print(f"fit on {len(fit_rows):,} inspections (2016 to 2023) | test on {len(test):,} | "
          "the four existing rows equal results/results.csv")

LASSO keeps 19 of 77 weights when refitted on 2016 to 2023


fit on 41,393 inspections (2016 to 2023) | test on 17,171 | the four existing rows equal results/results.csv


In [6]:
if FROZEN is not None:
    test_diffs = pd.concat([E.difference_table(y_test, test_scores, name, ["M1_lgbm", "B2_logreg"], groups=test_groups)
                            for name in NEW], ignore_index=True)
    test_diffs.insert(0, "split", "test")
    pd.concat([val_diffs, test_diffs], ignore_index=True).to_csv("../results/comparison_differences.csv", index=False)

    def settings_text(name):
        if name in NEW:
            return ", ".join(f"{k}={'no limit' if v is None else v}" for k, v in FROZEN[name]["params"].items())
        if name == "M1_lgbm":
            return ", ".join(f"{k}={FROZEN_M1['M1_lgbm'][k]}" for k in lgbm_keys)
        return {"B0_random": "none", "B1_past_fail_rate": "none (a rule)", "B2_logreg": "C=1 (default, not tuned)"}[name]

    vs_m1 = test_diffs[(test_diffs["minus"] == "M1_lgbm") & (test_diffs["metric"] == "precision_at_k")].set_index("model")
    test_results.insert(1, "settings", [settings_text(m) for m in test_results["model"]])
    test_results.insert(2, "settings_tried", [FROZEN[m]["settings_tried"] if m in NEW else {"M1_lgbm": 12}.get(m, 0)
                                              for m in test_results["model"]])
    test_results["val_precision_at_k"] = test_results["model"].map(val_results.set_index("model")["precision_at_k"])
    for col, source in [("minus_M1", "difference"), ("minus_M1_lo", "ci_lo"), ("minus_M1_hi", "ci_hi"), ("minus_M1_verdict", "verdict")]:
        test_results[col] = test_results["model"].map(vs_m1[source])
    test_results.to_csv("../results/comparison_test.csv", index=False)

    labels = {**E.MODEL_LABELS, **C.LABELS}
    show = pd.DataFrame({
        "model": [labels[m] for m in test_results["model"]],
        "settings": test_results["settings"],
        "2023 hit rate": (test_results["val_precision_at_k"] * 100).round(1),
        "test hit rate": (test_results["precision_at_k"] * 100).round(1),
        "95% range": [f"{lo * 100:.1f} to {hi * 100:.1f}" for lo, hi in zip(test_results["precision_at_k_lo"], test_results["precision_at_k_hi"])],
        "test catch": (test_results["catch_at_k"] * 100).round(1),
        "test AUC": test_results["auc"].round(3),
        "vs. LightGBM (points)": ["" if pd.isna(d) else f"{d * 100:+.1f} ({lo * 100:+.1f} to {hi * 100:+.1f}), {v}"
                                  for d, lo, hi, v in zip(test_results["minus_M1"], test_results["minus_M1_lo"],
                                                          test_results["minus_M1_hi"], test_results["minus_M1_verdict"])],
    })
    print(show.to_string(index=False))

                          model                                                settings  2023 hit rate  test hit rate    95% range  test catch  test AUC             vs. LightGBM (points)
   No ranking (reference point)                                                    none           55.8           59.0 57.8 to 59.9        20.0     0.500                                  
Past failure rate (simple rule)                                           none (a rule)           80.4           80.3 78.5 to 81.7        27.2     0.698                                  
            Logistic regression                                C=1 (default, not tuned)           88.9           87.0 85.2 to 88.2        29.5     0.736                                  
              LightGBM, history     num_leaves=31, learning_rate=0.02, n_estimators=231           90.5           88.1 86.4 to 89.4        29.9     0.748                                  
      Ridge logistic regression                                  

Hit rate: the share of the first 20% of visits that find a failure. Catch: the share of all failures those visits find. The 95% ranges resample whole restaurants. The last column is the paired difference from LightGBM (M1) on the test years, with the verdict wording fixed in `src/evaluate.py` before the first test scoring.

Every model reads the same 17 inputs, so a small spread between them says that the inspection history carries the signal and the choice of algorithm adds little.